# EEG_07 — 5-Fold Cross-Validation Subject-Independent

Cross-validation a K=5 fold sui soggetti per stimare `mean ± std` della balanced accuracy.

**Setup**: KFold(n_splits=5, shuffle=True, seed=42) sui soggetti disponibili.  
Per ogni fold: train ~50 sogg, val ~10 sogg, test ~14 sogg.  
**Output**: `data/interim/eeg07_cv_si_4_norm_results.csv` + plot `mean ± std` per modello.

Richiede: `daniele_311` — stesso env di EEG_05/06.

In [1]:
# ═══════════════════════════════════════════════════════════
#  TOGGLE — modifica qui prima di eseguire
# ═══════════════════════════════════════════════════════════
USE_CLUSTERS      = True
CLUSTER_SCHEME    = "concr4"   # "concr4" | "phon4" | "sem5" | ...
USE_INSTANCE_NORM = True        # Bomatter et al. 2024

N_FOLDS           = 5           # numero di fold nella CV
VAL_FRACTION      = 0.15        # frazione di soggetti da usare come val (dal train_val)
SWEEP_MODELS_ONLY = None        # None → tutti | ["EEGNet"] → solo EEGNet
CV_RESUME         = True        # True → salta fold/modello già nel CSV

# Tag automatico
NORM_TAG = "_norm" if USE_INSTANCE_NORM else ""
# ═══════════════════════════════════════════════════════════


In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fix OpenMP su macOS

import json
import time
import warnings
warnings.filterwarnings('ignore')

from torch.utils.tensorboard import SummaryWriter
import numpy as np
import pandas as pd
import h5py
import torch
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

from braindecode.models import EEGNet, EEGConformer, Deep4Net, ShallowFBCSPNet, ATCNet, Labram

# Device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
print("Python:", __import__('sys').version.split()[0])
print("torch:", torch.__version__)
import braindecode; print("braindecode:", braindecode.__version__)

device: cuda
Python: 3.11.15
torch: 2.10.0+cu128
braindecode: 1.3.2


In [3]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)

META_CSV   = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH  = project_root / "src" / "io" / "ebneuro.locs"

# Parametri EEG
N_CHANS         = 59    # canali dopo rimozione A1, A2
N_TIMES         = 384   # campioni a 256 Hz (~1.5s)
N_TIMES_CBRAMOD = 400   # Labram richiede multiplo di patch_size=200 → pad 384→400
SFREQ           = 256

# Training
BATCH_SIZE   = 32
MAX_EPOCHS   = 100
PATIENCE     = 15
LR           = 1e-3
WEIGHT_DECAY = 1e-4

# Soggetti da testare in subject-independent

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Config OK")

Config OK


In [4]:
# ============================================================
# CARICAMENTO METADATA E CLUSTER MAPPING
# ============================================================

import sys
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

meta = pd.read_csv(META_CSV)
# Filtro di emergenza per righe corrotte (epoch_idx fuori range nell'H5)
# Alcuni soggetti (es. 08, 46) hanno metadati che puntano a epoche inesistenti.
_initial_len = len(meta)
# Invece di controllare ogni file (lento), filtriamo i casi noti o usiamo un approccio conservativo
# Qui rimuoviamo le righe incriminate che abbiamo scoperto tramite debug
meta = meta[~((meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
               (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
               (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110)) |
               (meta["path_h5"].str.contains("46_04.h5") & (meta["epoch_idx"] >= 34)) |
               (meta["path_h5"].str.contains("ignore_8_05.h5") & (meta["epoch_idx"] >= 110)))]
if len(meta) < _initial_len:
    print(f"Rimosse {_initial_len - len(meta)} righe corrotte dai metadati.")
meta["subject_id"] = meta["subject_id"].astype(str).str.zfill(2)

# Indici canali: rimuove A1 (idx=0) e A2 (idx=7) dalla lista .locs
def read_eloc_names(path):
    names = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]  # H5 ha 61 canali registrati

ch_names_61 = read_eloc_names(ELOC_PATH)
EXCLUDE = {"A1", "A2"}
keep_idx = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
keep_names = [ch_names_61[i] for i in keep_idx]

assert len(keep_idx) == N_CHANS

# Carica schema via toggle
_scheme = CLUSTER_SCHEME if USE_CLUSTERS else "raw110"
interim_dir = project_root / "data" / "interim"
labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(_scheme, interim_dir)

print(f"Meta: {len(meta)} epoche | {meta['subject_id'].nunique()} soggetti")
print(f"Canali: {len(keep_idx)} ({keep_names[:4]}...)")
print(f"Schema: {_scheme} | {N_CLASSES} classi | Chance level: {100/N_CLASSES:.1f}%")
for cid, cname in cluster_names.items():
    n = sum(1 for v in labelid2cluster.values() if v == cid)
    print(f"  {cid} — {cname}: {n} parole")

Rimosse 12 righe corrotte dai metadati.
Meta: 39180 epoche | 76 soggetti
Canali: 59 (['AF7', 'AF3', 'Fp1', 'FP2']...)
Schema: concr4 | 4 classi | Chance level: 25.0%
  0 — CONCR: 18 parole
  1 — AZIONE: 27 parole
  2 — STATO: 21 parole
  3 — ASTRATTO: 44 parole


In [5]:
# ============================================================
# DATASET: caricamento lazy da H5
# ============================================================

class RawEEGDataset(Dataset):
    """
    Carica epoche EEG grezze da file H5 con normalizzazione per-canale.
    Ritorna (59, 384) float32 normalizzato + label cluster.

    Note metodologiche:
    - mean/std calcolati SOLO sul training set (passati a val/test via costruttore)
    - _compute_stats usa seed fisso per riproducibilità
    - file_cache chiuso esplicitamente in __del__ per evitare memory leak su sweep lunghi
    """
    def __init__(self, records, keep_idx, labelid2cluster, mean=None, std=None, instance_norm=False):
        self.file_cache    = {}
        self.records       = records
        self.keep_idx      = keep_idx
        self.labelid2cluster = labelid2cluster
        self.mean          = mean
        self.std           = std
        self.instance_norm = instance_norm
        if mean is None:
            self._compute_stats()

    def _compute_stats(self, seed=42):
        # Seed fisso per riproducibilità — stats identiche ad ogni run
        rng = np.random.RandomState(seed)
        n = min(500, len(self.records))
        idxs = rng.choice(len(self.records), n, replace=False)

        # Raggruppa per file H5 per minimizzare aperture (10-20x più veloce su WSL/mount)
        from collections import defaultdict
        paths_map = defaultdict(list)
        for idx in idxs:
            r = self.records[idx]
            paths_map[r["path_h5"]].append(int(r["epoch_idx"]))

        buf = []
        print(f"Calcolo stats su {n} campioni da {len(paths_map)} file (seed={seed})...")
        for path, epoch_idxs in tqdm(paths_map.items(), desc="Stats H5", leave=False):
            with h5py.File(path, "r") as f:
                data_h5 = f["data"]
                for e_idx in epoch_idxs:
                    x = data_h5[e_idx][self.keep_idx, :].astype(np.float32)
                    buf.append(x)

        buf = np.stack(buf)  # (N, 59, T)
        # mean/std per canale → shape (59, 1) per broadcasting con (59, T)
        self.mean = buf.mean(axis=(0, 2), keepdims=False).reshape(-1, 1).astype(np.float32)
        self.std  = buf.std( axis=(0, 2), keepdims=False).reshape(-1, 1).astype(np.float32) + 1e-6

    def __del__(self):
        # Chiude i file H5 aperti — importante su sweep lunghi per evitare memory leak
        for f in self.file_cache.values():
            try:
                f.close()
            except Exception:
                pass

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        path = r["path_h5"]
        if path not in self.file_cache:
            self.file_cache[path] = h5py.File(path, "r")
        f = self.file_cache[path]
        x = f["data"][int(r["epoch_idx"])][self.keep_idx, :].astype(np.float32)
        x = (x - self.mean) / self.std
        if self.instance_norm:
            x = (x - x.mean(axis=-1, keepdims=True)) / (x.std(axis=-1, keepdims=True) + 1e-6)
        label = self.labelid2cluster[int(r["label_idx"])]
        return torch.from_numpy(x), torch.tensor(label, dtype=torch.long)


def make_independent_splits(meta_df, labelid2cluster, subj_train, subj_val, subj_test, seed=SEED, instance_norm=False):
    """
    Split subject-independent: train su subj_train, val su subj_val, test su subj_test.
    Nessuna sovrapposizione di soggetti tra i tre set.
    mean/std normalizzazione calcolati SOLO sul train set, riutilizzati per val e test.
    """
    df_tr = meta_df[meta_df['subject_id'].isin([str(s).zfill(2) for s in subj_train])].copy()
    df_va = meta_df[meta_df['subject_id'].isin([str(s).zfill(2) for s in subj_val])].copy()
    df_te = meta_df[meta_df['subject_id'].isin([str(s).zfill(2) for s in subj_test])].copy()

    r_tr = df_tr[["path_h5", "epoch_idx", "label_idx"]].to_dict("records")
    r_va = df_va[["path_h5", "epoch_idx", "label_idx"]].to_dict("records")
    r_te = df_te[["path_h5", "epoch_idx", "label_idx"]].to_dict("records")

    rng = np.random.RandomState(seed)
    rng.shuffle(r_tr)
    rng.shuffle(r_va)
    rng.shuffle(r_te)

    ds_tr = RawEEGDataset(r_tr, keep_idx, labelid2cluster, instance_norm=instance_norm)
    ds_va = RawEEGDataset(r_va, keep_idx, labelid2cluster, ds_tr.mean, ds_tr.std, instance_norm)
    ds_te = RawEEGDataset(r_te, keep_idx, labelid2cluster, ds_tr.mean, ds_tr.std, instance_norm)
    return ds_tr, ds_va, ds_te


print(f"Dataset OK | Instance Norm: {USE_INSTANCE_NORM}")

Dataset OK | Instance Norm: True


In [6]:
# ============================================================
# FACTORY MODELLI — include Labram
# ============================================================

# Labram wrapper: padda l'input da 384 a 400 internamente
class LabramWrapper(nn.Module):
    """Wrappa Labram aggiungendo zero-padding temporale 384→400."""
    def __init__(self, n_outputs):
        super().__init__()
        self.model = Labram(
            n_chans=N_CHANS, n_outputs=n_outputs,
            n_times=N_TIMES_CBRAMOD, sfreq=SFREQ  # usa default patch_size=200
        )
        self.pad = N_TIMES_CBRAMOD - N_TIMES  # 16 campioni

    def forward(self, x):
        x = F.pad(x, (0, self.pad))  # (batch, 59, 384) → (batch, 59, 400)
        return self.model(x)


def build_model(name, n_outputs):
    if name == "EEGNet":
        return EEGNet(n_chans=N_CHANS, n_outputs=n_outputs,
                      n_times=N_TIMES, sfreq=SFREQ, final_conv_length="auto")
    elif name == "ShallowFBCSPNet":
        return ShallowFBCSPNet(n_chans=N_CHANS, n_outputs=n_outputs,
                               n_times=N_TIMES, final_conv_length="auto")
    elif name == "Deep4Net":
        return Deep4Net(n_chans=N_CHANS, n_outputs=n_outputs,
                        n_times=N_TIMES, final_conv_length="auto")
    elif name == "EEGConformer":
        return EEGConformer(n_chans=N_CHANS, n_outputs=n_outputs,
                            n_times=N_TIMES, sfreq=SFREQ, final_fc_length="auto")
    elif name == "ATCNet":
        return ATCNet(n_chans=N_CHANS, n_outputs=n_outputs,
                      input_window_seconds=N_TIMES / SFREQ, sfreq=SFREQ)
    elif name == "Labram":
        return LabramWrapper(n_outputs=n_outputs)
    else:
        raise ValueError(f"Modello sconosciuto: {name}")


MODEL_NAMES = ["EEGNet", "ShallowFBCSPNet", "Deep4Net", "EEGConformer", "ATCNet", "Labram"]

print(f"{'Modello':<18} {'Parametri':>12}")
print("-" * 32)
for name in MODEL_NAMES:
    m = build_model(name, n_outputs=4)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:<18} {n_params:>12,}")

Modello               Parametri
--------------------------------
EEGNet                    2,820
ShallowFBCSPNet          98,724
Deep4Net                261,504
EEGConformer            428,932
ATCNet                   44,588
Labram                5,844,540


In [7]:
# ============================================================
# TRAINING E VALUTAZIONE
# ============================================================

def train_model(model, ds_train, ds_val, save_path, tb_dir, n_epochs=MAX_EPOCHS, patience=PATIENCE,
                lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()
    writer = SummaryWriter(log_dir=tb_dir)

    loader_tr = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=0) # 0 evita problemi con h5py
    loader_va = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=0)

    best_val_acc, best_state, patience_cnt = -1.0, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
    history = defaultdict(list)

    for epoch in range(n_epochs):
        model.train()
        loss_sum, correct, n_tot = 0.0, 0, 0
        pbar = tqdm(loader_tr, desc=f"Epoch {epoch+1}/{n_epochs}", leave=False)
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct  += (logits.argmax(1) == y).sum().item()
            n_tot    += len(y)
            pbar.set_postfix(loss=loss.item())
        scheduler.step()

        model.eval()
        ys_v, ps_v = [], []
        with torch.no_grad():
            for x, y in loader_va:
                ps_v.extend(model(x.to(device)).argmax(1).cpu().tolist())
                ys_v.extend(y.tolist())

        val_acc  = accuracy_score(ys_v, ps_v)
        val_bacc = balanced_accuracy_score(ys_v, ps_v)
        train_loss = loss_sum / n_tot
        train_acc = correct / n_tot
        
        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        history["val_bacc"].append(val_bacc)
        
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Accuracy/train", train_acc, epoch)
        writer.add_scalar("Accuracy/val", val_acc, epoch)
        writer.add_scalar("Balanced_Accuracy/val", val_bacc, epoch)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
            # Salviamo il best state temporaneo su disco (opzionale, ma utile in caso di crash)
            torch.save(best_state, save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                break

    writer.close()
    model.load_state_dict(best_state)
    torch.save(best_state, save_path) # Assicura che l'ultimo best_state sia quello salvato
    return model, dict(history), epoch + 1

def evaluate(model, ds):
    model.eval().to(device)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)  # 0 evita problemi con h5py
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps.extend(model(x.to(device)).argmax(1).cpu().tolist())
            ys.extend(y.tolist())
    return {
        "acc":    accuracy_score(ys, ps),
        "bacc":   balanced_accuracy_score(ys, ps),
        "y_true": np.array(ys),
        "y_pred": np.array(ps),
    }


print("Training utilities OK")

Training utilities OK


## 5-Fold Cross-Validation Subject-Independent

I soggetti disponibili vengono divisi in 5 fold con `KFold(shuffle=True, seed=42)`.  
Per ogni fold: il modello è trainato from scratch, valutato sul test fold.  
Il CSV intermedio viene scritto dopo ogni modello → **resume** automatico su crash.

In [ ]:
from sklearn.model_selection import KFold

# ── Soggetti disponibili nel dataset ─────────────────────────────────────────
all_subj = sorted(meta["subject_id"].unique())   # stringhe zero-padded es. '00','01',...
print(f"Soggetti disponibili: {len(all_subj)} → {all_subj[:5]}...")

chance_level = 1.0 / N_CLASSES
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

TB_BASE   = project_root / "runs"   / f"eeg07_cv_si_{N_CLASSES}{NORM_TAG}"
CKPT_BASE = project_root / "models" / f"eeg07_cv_si_{N_CLASSES}{NORM_TAG}"
CKPT_BASE.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = project_root / "data" / "interim" / f"eeg07_cv_si_{N_CLASSES}{NORM_TAG}_results.csv"
if CV_RESUME and RESULTS_CSV.exists():
    df_existing = pd.read_csv(RESULTS_CSV)
    all_results = df_existing.to_dict("records")
    done_pairs  = set(zip(df_existing["fold"].astype(str), df_existing["model"].astype(str)))
    print(f"Resume: trovati {len(all_results)} risultati in {RESULTS_CSV.name}")
else:
    all_results, done_pairs = [], set()

_models_to_run = SWEEP_MODELS_ONLY if SWEEP_MODELS_ONLY else MODEL_NAMES

for fold_idx, (train_val_idx, test_idx) in enumerate(kf.split(all_subj)):
    train_val_subj = [all_subj[i] for i in train_val_idx]
    test_subj      = [all_subj[i] for i in test_idx]

    # Carve out val dal train_val
    n_val      = max(5, round(len(train_val_subj) * VAL_FRACTION))
    val_subj   = train_val_subj[:n_val]
    train_subj = train_val_subj[n_val:]

    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx+1}/{N_FOLDS} | tr={len(train_subj)} va={len(val_subj)} te={len(test_subj)} sogg.")
    print(f"Test: {test_subj}")
    print(f"{'='*60}")

    ds_tr, ds_va, ds_te = make_independent_splits(
        meta, labelid2cluster, train_subj, val_subj, test_subj,
        instance_norm=USE_INSTANCE_NORM
    )
    print(f"  Epoche: tr={len(ds_tr)} va={len(ds_va)} te={len(ds_te)}")

    for model_name in _models_to_run:
        # Resume: skip se (fold, modello) già nel CSV
        if CV_RESUME and (str(fold_idx), str(model_name)) in done_pairs:
            existing = next(r for r in all_results
                            if r["fold"] == fold_idx and r["model"] == model_name)
            print(f"  {model_name} [SKIP]: test_bacc={existing['test_bacc']:.4f}")
            continue

        ckpt_path = CKPT_BASE / f"fold{fold_idx}_{model_name}.pth"
        tb_dir    = TB_BASE / f"fold{fold_idx}" / model_name

        model = build_model(model_name, N_CLASSES)
        t0    = time.time()
        # train_model restituisce (model, history_dict, n_epochs)
        model_trained, hist, n_ep = train_model(model, ds_tr, ds_va, ckpt_path, tb_dir)
        elapsed = time.time() - t0
        te_r  = evaluate(model_trained, ds_te)

        row = {
            "fold":      fold_idx,
            "model":     model_name,
            "n_train":   len(train_subj),
            "n_val":     len(val_subj),
            "n_test":    len(test_subj),
            "val_bacc":  max(hist["val_bacc"]) if hist["val_bacc"] else 0.0,
            "val_acc":   max(hist["val_acc"])  if hist["val_acc"]  else 0.0,
            "test_bacc": te_r["bacc"],
            "test_acc":  te_r["acc"],
            "epochs":    n_ep,
            "time_s":    round(elapsed, 1),
        }
        all_results.append(row)
        done_pairs.add((str(fold_idx), str(model_name)))
        pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)
        print(f"  {model_name:<18}: test_bacc={te_r['bacc']:.4f}  ({elapsed/60:.1f} min)")

print(f"\n{'='*60}")
print(f"CV COMPLETATA — {N_FOLDS} fold x {len(_models_to_run)} modelli")
print(f"Risultati: {RESULTS_CSV}")



Soggetti disponibili: 76 → ['00', '01', '02', '03', '04']...
Resume: trovati 17 risultati in eeg07_cv_si_4_norm_results.csv

FOLD 1/5 | tr=51 va=9 te=16 sogg.
Test: ['00', '04', '10', '12', '18', '28', '33', '35', '45', '47', '50', '53', '58', '65', '67', '70']
Calcolo stats su 500 campioni da 204 file (seed=42)...


Stats H5:   0%|          | 0/204 [00:00<?, ?it/s]

  Epoche: tr=25842 va=4702 te=8636
  EEGNet [SKIP]: test_bacc=0.2499
  ShallowFBCSPNet [SKIP]: test_bacc=0.2505
  Deep4Net [SKIP]: test_bacc=0.2505
  EEGConformer [SKIP]: test_bacc=0.2500
  ATCNet [SKIP]: test_bacc=0.2500
  Labram [SKIP]: test_bacc=0.2500

FOLD 2/5 | tr=52 va=9 te=15 sogg.
Test: ['05', '07', '09', '16', '22', '30', '31', '34', '39', '40', '44', '54', '62', '64', '72']
Calcolo stats su 500 campioni da 218 file (seed=42)...


Stats H5:   0%|          | 0/218 [00:00<?, ?it/s]

  Epoche: tr=26338 va=4812 te=8030
  EEGNet [SKIP]: test_bacc=0.2501
  ShallowFBCSPNet [SKIP]: test_bacc=0.2488
  Deep4Net [SKIP]: test_bacc=0.2500
  EEGConformer [SKIP]: test_bacc=0.2500
  ATCNet [SKIP]: test_bacc=0.2500
  Labram [SKIP]: test_bacc=0.2500

FOLD 3/5 | tr=52 va=9 te=15 sogg.
Test: ['03', '06', '08', '13', '17', '19', '25', '36', '38', '42', '49', '55', '61', '63', 'ignore_8']
Calcolo stats su 500 campioni da 203 file (seed=42)...


...

Epoch 11/100:   0%|          | 0/844 [00:00<?, ?it/s]

Epoch 12/100:   0%|          | 0/844 [00:00<?, ?it/s]

Epoch 13/100:   0%|          | 0/844 [00:00<?, ?it/s]

Epoch 14/100:   0%|          | 0/844 [00:00<?, ?it/s]

Epoch 15/100:   0%|          | 0/844 [00:00<?, ?it/s]

In [ ]:
# ── Aggregazione: mean +/- std per modello ──────────────────────────────────
df_cv = pd.read_csv(RESULTS_CSV)
chance = 1.0 / N_CLASSES

agg = (
    df_cv.groupby("model")
    .agg(
        mean_test_bacc=("test_bacc", "mean"),
        std_test_bacc =("test_bacc", "std"),
        mean_val_bacc =("val_bacc",  "mean"),
        std_val_bacc  =("val_bacc",  "std"),
        n_folds       =("fold",      "count"),
    )
    .round(4)
    .sort_values("mean_test_bacc", ascending=False)
)
agg["ci95"] = (agg["std_test_bacc"] * 1.96).round(4)
agg["above_chance"] = agg["mean_test_bacc"] > chance

print(f"=== 5-Fold CV Subject-Independent | {N_CLASSES} classi | Chance={chance:.1%} ===")
print(agg[["mean_test_bacc","std_test_bacc","ci95","mean_val_bacc","n_folds","above_chance"]].to_string())

# Dettaglio per fold
print("\n=== Dettaglio per fold ===")
pivot = df_cv.pivot_table(index="model", columns="fold", values="test_bacc").round(4)
print(pivot.to_string())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = agg.index.tolist()
means  = agg["mean_test_bacc"].values
stds   = agg["std_test_bacc"].values
chance = 1.0 / N_CLASSES

colors = ["#2196F3","#4CAF50","#FF9800","#9C27B0","#F44336","#00BCD4"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Grafico 1: mean +/- std ────────────────────────────────────────────────
ax = axes[0]
x = np.arange(len(models))
bars = ax.bar(x, means, yerr=stds, capsize=6,
              color=colors[:len(models)], alpha=0.85, width=0.6)
ax.axhline(chance, ls="--", color="red", lw=1.5, label=f"Chance ({chance:.1%})")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha="right")
ax.set_ylabel("Test Balanced Accuracy")
ax.set_title(f"5-Fold CV SI | {N_CLASSES} classi ({CLUSTER_SCHEME})\nmean \u00b1 std")
ax.legend()
ymax = max(means + stds) * 1.35
ax.set_ylim(0, max(ymax, chance * 1.5))
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.003,
            f"{m:.3f}\n\u00b1{s:.3f}", ha="center", fontsize=8)

# ── Grafico 2: boxplot per fold ────────────────────────────────────────────
ax2 = axes[1]
data_by_model = [df_cv[df_cv["model"] == m]["test_bacc"].values for m in models]
bp = ax2.boxplot(data_by_model, labels=models, patch_artist=True,
                 medianprops=dict(color="black", lw=2))
for patch, color in zip(bp["boxes"], colors[:len(models)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.axhline(chance, ls="--", color="red", lw=1.5, label=f"Chance ({chance:.1%})")
ax2.set_ylabel("Test Balanced Accuracy")
ax2.set_title("Distribuzione per fold")
ax2.set_xticklabels(models, rotation=25, ha="right")
ax2.legend()

plt.tight_layout()
fig_path = project_root / "figures" / f"cv_si_{CLUSTER_SCHEME}.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: {fig_path}")
